### Configure our storage paths

In [0]:
# Get the current catalog
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "shopsphere"

# Source location
BASE_PATH = f"/Volumes/{catalog}/{schema_name}/raw_source"

# Bronze table namespace
BRONZE_SCHEMA = schema_name

print("Catalog:", catalog)
print("Schema:", BRONZE_SCHEMA)
print("Source:", BASE_PATH)

Define our datasets

In [0]:
source_datasets = {
    "customers": f"{BASE_PATH}/customers",
    "products": f"{BASE_PATH}/products",
    "orders": f"{BASE_PATH}/orders",
    "order_items": f"{BASE_PATH}/order_items",
    "payments": f"{BASE_PATH}/payments"
}

for name, path in source_datasets.items():
    print(name, "->", path)

Read CSV files using Spark

In [0]:
raw_dataframes = {}

for name, path in source_datasets.items():

    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .load(path)
    )

    raw_dataframes[name] = df

    print(f"Loaded {name}: {df.count()} records")

Add ingestion metadata

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)

bronze_dataframes = {}

for name, df in raw_dataframes.items():

    bronze_df = (
        df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_source_system", lit("ShopSphere"))
    )

    bronze_dataframes[name] = bronze_df

    print(f"Metadata added to {name}")

Write Bronze Delta tables

In [0]:
for name, df in bronze_dataframes.items():

    table_name = f"bronze_{name}"

    full_table_name = (
        f"`{catalog}`.`{BRONZE_SCHEMA}`.`{table_name}`"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )

    print(f"Created table: {full_table_name}")

Verify our Bronze tables

In [0]:
for name in source_datasets.keys():

    table_name = f"bronze_{name}"

    full_table_name = (
        f"`{catalog}`.`{BRONZE_SCHEMA}`.`{table_name}`"
    )

    df = spark.table(full_table_name)

    print(f"\nTable: {table_name}")
    print("Records:", df.count())

    df.printSchema()
    df.show(5, truncate=False)